In [1]:
# CELL 1: Student Configuration

LAST_NAME = "Gabilan"
STUDENT_ID = "TUPM-26-2636"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(digit) for digit in STUDENT_ID if digit.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)

Student: Gabilan
Student ID: TUPM-26-2636
Seed Digit: 6
ID Sum: 25
Name Length: 7


In [2]:
import csv
import json
import shutil
import os
from collections import Counter

In [3]:
INPUT_FILE = "weblog.csv"

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILTERED_FILE = os.path.join(OUTPUT_DIR, "filtered_web_logs.csv")
SUMMARY_FILE = os.path.join(OUTPUT_DIR, "web_log_summary.json")
BACKUP_FILE = os.path.join(OUTPUT_DIR, "web_log_summary_backup.json")

RECORD_LIMIT = 1000 + ID_SUM
TARGET_CLASS = (SEED_DIGIT % 4) + 2

print("Records to Process:", RECORD_LIMIT)
print("Target Status Class:", f"{TARGET_CLASS}xx")

Records to Process: 1025
Target Status Class: 4xx


In [4]:
records = []
invalid_records = 0
total_records = 0

with open(INPUT_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_records += 1

        if total_records > RECORD_LIMIT:
            break

        try:
            ip = row["IP"].strip()
            timestamp = row["Time"].strip()
            url = row["URL"].strip()
            status = int(row["Staus"])

            if not ip or not timestamp or not url or not (100 <= status <= 599):
                invalid_records += 1
                continue

            records.append({
                "IP": ip,
                "Time": timestamp,
                "URL": url,
                "Status": status
            })

        except (KeyError, ValueError, AttributeError):
            invalid_records += 1

print("Valid Records:", len(records))
print("Invalid Records:", invalid_records)

Valid Records: 1025
Invalid Records: 0


In [5]:
target_records = [
    record for record in records
    if record["Status"] // 100 == TARGET_CLASS
]

print("Target Status Class:", f"{TARGET_CLASS}xx")
print("Records in Target Class:", len(target_records))

Target Status Class: 4xx
Records in Target Class: 5


In [6]:
status_counts = Counter(
    record["Status"] for record in target_records
)

url_counts = Counter(
    record["URL"] for record in target_records
)

top_five_urls = url_counts.most_common(5)

print("Count per Status Code:")
for status, count in sorted(status_counts.items()):
    print(status, ":", count)

print("\nFive Most Requested URLs:")
for url, count in top_five_urls:
    print(url, ":", count)

Count per Status Code:
404 : 5

Five Most Requested URLs:
GET /robots.txt HTTP/1.1 : 4
GET /favicon.ico HTTP/1.1 : 1


In [7]:
with open(FILTERED_FILE, "w", newline="", encoding="utf-8") as file:
    fieldnames = ["IP", "Time", "URL", "Status"]
    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(target_records)

print("Filtered CSV saved:", FILTERED_FILE)

Filtered CSV saved: output/filtered_web_logs.csv


In [8]:
summary = {
    "Student": LAST_NAME,
    "Student ID": STUDENT_ID,
    "Records Processed": len(records),
    "Target Status Class": f"{TARGET_CLASS}xx",
    "Records in Target Class": len(target_records),
    "Count per Status Code": dict(status_counts),
    "Five Most Requested URLs": [
        {"URL": url, "Count": count}
        for url, count in top_five_urls
    ],
    "Invalid Records": invalid_records
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON Summary saved:", SUMMARY_FILE)

JSON Summary saved: output/web_log_summary.json


In [9]:
shutil.copy2(SUMMARY_FILE, BACKUP_FILE)

print("Backup Created:", os.path.exists(BACKUP_FILE))

if os.path.exists(BACKUP_FILE):
    print("Backup File Size:", os.path.getsize(BACKUP_FILE), "bytes")

Backup Created: True
Backup File Size: 465 bytes


In [10]:
print("===== WEB SERVER ACCESS LOG REPORT =====")
print("Records Processed:", len(records))
print("Target Status Class:", f"{TARGET_CLASS}xx")
print("Records in Target Class:", len(target_records))
print("Filtered CSV:", FILTERED_FILE)
print("JSON Summary:", SUMMARY_FILE)
print("Backup Verified:", os.path.exists(BACKUP_FILE))
print("========================================")

===== WEB SERVER ACCESS LOG REPORT =====
Records Processed: 1025
Target Status Class: 4xx
Records in Target Class: 5
Filtered CSV: output/filtered_web_logs.csv
JSON Summary: output/web_log_summary.json
Backup Verified: True
